# Prediction diagnostics: real surfacings vs. model predictions

For every scored surfacing, four views of the same event:

1. **Position** -- where each model predicted the float would surface vs.
   where it actually did, with a plain-English explanation of the error.
2. **Map** -- all floats' real surfacing tracks with predicted positions
   overlaid, per `docs/map.html`'s own overlay convention.
3. **Timing** -- when each model expected the float to surface vs. when it
   really did, and how that estimate evolved as fresher forecast data
   arrived.
4. **Depth** -- the simulated dive profile (descent/park/ascent, from the
   float's `cycle_action`) vs. the float's real recorded pressure trace for
   the same dive, where cached real data is fresh enough to compare.

Reuses the pipeline's own load functions (`src/float_store.py`) and model
color palette (`src/web_export.py`) rather than re-deriving anything from
raw parquet.

**Run `git pull` first** so `data/store/*.parquet` reflects the latest
scored surfacings. For section 4 (depth), also run the normal pipeline
(`python main.py`) beforehand if you can -- it refreshes
`data/argo_cache/*_Rtraj.nc` with `force_refresh=True` every run (see
`run._refresh_cycle_actions`), and that cache is gitignored, so it only
has data if something on this machine has fetched it. A stale cache is
handled gracefully (skipped with a reason), not silently plotted as if
it were current.


In [ ]:
from pathlib import Path
from datetime import timedelta
import math

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from src import float_store, run, web_export
from src.simulate import ControlAction

cfg = run._load_config()
run._build_globals(cfg)

# run.STORE_DIR/ARGO_CACHE_DIR come from config.toml as relative paths, and
# float_store's load_* functions just check Path(...).exists() with no
# fallback -- if this notebook's working directory isn't the repo root (e.g.
# Jupyter launched from inside notebooks/), that resolves to nothing and
# every load comes back empty with no error. Anchor to the repo root via
# run.py's own file location instead of cwd.
PROJECT_ROOT   = Path(run.__file__).resolve().parent.parent
STORE_DIR      = PROJECT_ROOT / cfg["paths"]["store_dir"]
ARGO_CACHE_DIR = PROJECT_ROOT / cfg["paths"]["argo_cache_dir"]
assert STORE_DIR.exists(), f"STORE_DIR not found at {STORE_DIR}"

MODEL_COLOR  = web_export.MODEL_COLOR      # {"cmems": "#00D4FF", "fcoo": "#FF8C42"} -- same as the live map
MODEL_MARKER = {"cmems": "o", "fcoo": "s"}

# Status colors for the error-verdict bar in section 1 -- deliberately a
# different palette from MODEL_COLOR so "which model" and "how good" are
# never confused. Never relied on alone: every bar also gets a text label.
VERDICT_COLOR = {"beat": "#2E7D32", "matched": "#F9A825", "worse": "#C62828"}


In [ ]:
floats_db                = float_store.load_floats_db(STORE_DIR)
error_db                 = float_store.load_error_db(STORE_DIR)
forecast_history_db      = float_store.load_forecast_history(STORE_DIR)
cycle_action_history_db  = float_store.load_cycle_action_history(STORE_DIR)
leg_history_db           = float_store.load_leg_history(STORE_DIR)

# One row per scored surfacing per model, real + predicted position both
# present -- exactly what web_export._build_scoring_history draws lines
# from. Older rows saved before this schema existed have NaN real_lat and
# are dropped here, same as web_export does.
scoring = error_db.dropna(subset=["real_lat", "predicted_lat"]).copy()
scoring["t"] = pd.to_datetime(scoring["t"])

print(f"{len(floats_db)} floats loaded, {len(scoring)} scored surfacings with a real+predicted pair")


## 1. Every predicted surfacing vs. real, explained

For each scored surfacing: the straight-line error, the compass direction
the prediction missed by, and the error expressed as a percentage of how
far the float actually drifted that leg (`error_pct`). That percentage is
the useful number -- a 5 km miss is meaningless without knowing whether the
float drifted 2 km or 200 km that leg. `error_pct >= 100%` means the model
did no better than a naive "the float didn't move" guess (a *persistence*
baseline); this is the bar you'd want a forecast to clear.


In [ ]:
def _bearing_deg(lat1, lon1, lat2, lon2):
    """Compass bearing from (lat1,lon1) to (lat2,lon2), 0=N/90=E."""
    lat1r, lat2r = math.radians(lat1), math.radians(lat2)
    dlon = math.radians(lon2 - lon1)
    x = math.sin(dlon) * math.cos(lat2r)
    y = math.cos(lat1r) * math.sin(lat2r) - math.sin(lat1r) * math.cos(lat2r) * math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


_COMPASS = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
            "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]


def _compass_label(deg):
    return _COMPASS[round(deg / 22.5) % 16]


def _verdict(error_pct):
    if pd.isna(error_pct):
        return None
    if error_pct < 50:
        return "beat"
    if error_pct < 100:
        return "matched"
    return "worse"


_VERDICT_TEXT = {
    "beat":    "clearly beat a 'the float didn't move' guess",
    "matched": "roughly matched a 'the float didn't move' guess",
    "worse":   "did no better than assuming the float wouldn't move at all",
}


def _explain(r):
    parts = [
        f"{r['model']} placed float {r['float_id']}'s {r['t']:%Y-%m-%d %H:%M} "
        f"surfacing {r['error_km']:.1f} km away, to the {r['compass']}."
    ]
    if pd.notna(r["drift_km"]):
        parts.append(f"The float itself drifted {r['drift_km']:.1f} km since its previous surfacing.")
        if pd.notna(r["verdict"]):
            parts.append(f"Error was {r['error_pct']:.0f}% of that drift -- {_VERDICT_TEXT[r['verdict']]}.")
    else:
        parts.append("No drift figure recorded for this leg (older row, predates drift_m tracking).")
    return " ".join(parts)


scoring["error_km"]  = scoring["error_m"] / 1000.0
scoring["drift_km"]  = scoring["drift_m"] / 1000.0
# MIN_DRIFT_M_FOR_PCT mirrors web_export.export_leaderboard's own cutoff --
# below this the float barely moved and error/drift blows up on noise.
scoring["error_pct"] = np.where(
    scoring["drift_m"] >= web_export.MIN_DRIFT_M_FOR_PCT,
    scoring["error_m"] / scoring["drift_m"] * 100.0,
    np.nan,
)
scoring["bearing_deg"] = [
    _bearing_deg(r.real_lat, r.real_lon, r.predicted_lat, r.predicted_lon)
    for r in scoring.itertuples()
]
scoring["compass"] = scoring["bearing_deg"].apply(_compass_label)
scoring["verdict"] = scoring["error_pct"].apply(_verdict)
scoring["explanation"] = scoring.apply(_explain, axis=1)
scoring = scoring.sort_values("t", ascending=False).reset_index(drop=True)

display_cols = ["float_id", "model", "t", "error_km", "drift_km", "error_pct", "compass", "verdict"]
scoring[display_cols]


In [ ]:
for _, r in scoring.iterrows():
    print(f"- {r['explanation']}")


In [ ]:
# Horizontal bar of every scored event's error, colored by verdict against
# the persistence baseline -- color is never the only channel carrying that
# verdict, each bar also gets its percentage as a direct label.
plot_df = scoring.sort_values("t").reset_index(drop=True)
labels = [f"{r.float_id} / {r.model}\n{r.t:%m-%d %H:%M}" for r in plot_df.itertuples()]
colors = [VERDICT_COLOR.get(v, "0.6") for v in plot_df["verdict"]]

fig, ax = plt.subplots(figsize=(7, 0.4 * len(plot_df) + 1.5))
bars = ax.barh(labels, plot_df["error_km"], color=colors, height=0.6)
for bar, r in zip(bars, plot_df.itertuples()):
    label = f"{r.error_km:.1f} km"
    if pd.notna(r.error_pct):
        label += f"  ({r.error_pct:.0f}% of drift)"
    ax.text(bar.get_width() + max(plot_df["error_km"]) * 0.01, bar.get_y() + bar.get_height() / 2,
            label, va="center", fontsize=8)

ax.set_xlabel("Position error (km)")
ax.set_title("Every scored surfacing: prediction error vs. persistence baseline")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in VERDICT_COLOR.values()]
ax.legend(handles, [
    "beat baseline (<50% of drift)",
    "roughly matched baseline (50-100%)",
    "worse than baseline (>=100%)",
], loc="lower right", fontsize=8, framealpha=0.9)
ax.set_xlim(0, plot_df["error_km"].max() * 1.35 if len(plot_df) else 1)
fig.tight_layout()
plt.show()


## 2. Map: real surfacing tracks with predicted positions overlaid

Same overlay `docs/map.html` draws for a single selected float
(`drawFloatOverlays`), laid out for all floats at once. Dot color = real
surfacing time (each float has its own color scale, since floats were
registered at different times). Diamond/square markers = each model's
predicted position at a scored surfacing; thin colored line back to the
real position = the error, labeled with its distance and compass bearing.


In [ ]:
float_ids = sorted(floats_db.keys())
n = len(float_ids)
ncols = 3
nrows = -(-n // ncols)  # ceil

fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 5 * nrows), squeeze=False)

for idx, float_id in enumerate(float_ids):
    ax = axes[idx // ncols][idx % ncols]
    frow = floats_db[float_id]
    surf = sorted(frow.surfacing_history, key=lambda p: p[2])  # (lat, lon, t)

    if not surf:
        ax.text(0.5, 0.5, "no surfacing history yet", ha="center", va="center",
                transform=ax.transAxes, color="0.5")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{float_id} -- 0 surfacings")
        continue

    lats = [p[0] for p in surf]
    lons = [p[1] for p in surf]
    t_nums = mdates.date2num([p[2] for p in surf])

    ax.plot(lons, lats, "-", color="0.7", linewidth=0.8, zorder=1)
    if len(surf) >= 2:
        sc = ax.scatter(lons, lats, c=t_nums, cmap="viridis", s=32,
                         edgecolors="k", linewidths=0.3, zorder=3)
        cb = fig.colorbar(sc, ax=ax, shrink=0.8)
        cb.ax.yaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    else:
        ax.scatter(lons, lats, color="tab:blue", s=32, edgecolors="k", linewidths=0.3, zorder=3)

    fscoring = scoring[scoring["float_id"] == float_id]
    for r in fscoring.itertuples():
        color = MODEL_COLOR.get(r.model, "#888888")
        marker = MODEL_MARKER.get(r.model, "^")
        ax.plot([r.real_lon, r.predicted_lon], [r.real_lat, r.predicted_lat],
                "-", color=color, linewidth=1.0, alpha=0.75, zorder=2)
        ax.scatter([r.predicted_lon], [r.predicted_lat], color=color, marker=marker,
                   s=30, edgecolors="k", linewidths=0.3, zorder=4)
        ax.annotate(f"{r.error_km:.1f}km {r.compass}", (r.predicted_lon, r.predicted_lat),
                    fontsize=6.5, color=color, xytext=(3, 3), textcoords="offset points")

    mean_lat = np.mean(lats)
    ax.set_aspect(1.0 / max(math.cos(math.radians(mean_lat)), 1e-6))
    status = " (dead)" if frow.is_dead else ""
    ax.set_title(f"{float_id}{status} -- {len(surf)} surfacings, {len(fscoring)} scored")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")

for idx in range(n, nrows * ncols):
    axes[idx // ncols][idx % ncols].axis("off")

handles = [plt.Line2D([0], [0], marker=MODEL_MARKER.get(m, "^"), color=c, linestyle="",
                       markeredgecolor="k", markeredgewidth=0.3, markersize=7)
           for m, c in MODEL_COLOR.items()]
fig.legend(handles, [web_export.MODEL_DISPLAY.get(m, m) for m in MODEL_COLOR], loc="upper center",
           ncol=len(MODEL_COLOR), bbox_to_anchor=(0.5, 0.995), fontsize=9)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


## 3. Timing: when a surfacing was expected vs. when it happened

`forecast_history.parquet` records, for every pipeline run, what that
run's trajectory predicted as the *next* surfacing time. For each real
leg (the span between two consecutive confirmed real surfacings), this
matches every forecast issued during that leg to it, and plots how the
predicted surfacing time's error (`timing_error_h` = predicted minus
actual, hours) evolved as the forecast's lead time (hours before the real
surfacing) shrank. A model with real skill should converge toward 0 as
lead time shrinks; one that doesn't isn't using fresher data to sharpen
its estimate.


In [ ]:
fh = forecast_history_db.copy()
fh["forecast"] = pd.to_datetime(fh["forecast"])
fh["expected_surfacing_time"] = pd.to_datetime(fh["expected_surfacing_time"])

timing_rows = []
for float_id, frow in floats_db.items():
    real_times = [p[2] for p in sorted(frow.surfacing_history, key=lambda p: p[2])]
    for i in range(1, len(real_times)):
        prev_t, real_t = real_times[i - 1], real_times[i]
        leg_fh = fh[
            (fh["float_id"] == float_id) & (fh["forecast"] >= prev_t) & (fh["forecast"] < real_t)
        ].sort_values("forecast")
        for r in leg_fh.itertuples():
            timing_rows.append(dict(
                float_id=float_id, model=r.forecast_name, real_t=real_t,
                issue_time=r.forecast,
                lead_h=(real_t - r.forecast).total_seconds() / 3600.0,
                timing_error_h=(r.expected_surfacing_time - real_t).total_seconds() / 3600.0,
            ))

timing_df = pd.DataFrame(timing_rows)
print(f"{len(timing_df)} forecast-vs-real-surfacing timing rows across "
      f"{timing_df[['float_id','model','real_t']].drop_duplicates().shape[0] if len(timing_df) else 0} legs")
timing_df.sort_values(["float_id", "real_t", "issue_time"]) if len(timing_df) else timing_df


In [ ]:
if timing_df.empty:
    print("No forecast_history rows fall inside a completed real leg yet -- nothing to plot.")
else:
    legs = sorted(timing_df[["float_id", "real_t"]].drop_duplicates().itertuples(index=False),
                  key=lambda x: (x.float_id, x.real_t))
    ncols = 3
    nrows = -(-len(legs) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.3 * nrows), squeeze=False)

    for idx, leg in enumerate(legs):
        ax = axes[idx // ncols][idx % ncols]
        leg_df = timing_df[(timing_df["float_id"] == leg.float_id) & (timing_df["real_t"] == leg.real_t)]
        for model, mdf in leg_df.groupby("model"):
            mdf = mdf.sort_values("lead_h")
            ax.plot(mdf["lead_h"], mdf["timing_error_h"], "-o", color=MODEL_COLOR.get(model, "0.5"),
                    label=web_export.MODEL_DISPLAY.get(model, model), markersize=4)
        ax.axhline(0, color="0.3", linewidth=1, linestyle="--")
        ax.invert_xaxis()  # lead time shrinks left-to-right, ending at the real surfacing
        ax.set_title(f"{leg.float_id}\n{leg.real_t:%Y-%m-%d %H:%M}", fontsize=9)
        ax.set_xlabel("hours before real surfacing", fontsize=8)
        ax.set_ylabel("predicted - actual (h)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=7)

    for idx in range(len(legs), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    fig.suptitle("Timing error vs. lead time, per leg (0 = perfect, converging right = improving with fresher data)", fontsize=10)
    fig.tight_layout()
    plt.show()


In [ ]:
if not timing_df.empty:
    # The single most-recent (smallest lead_h) forecast per leg -- how far off
    # was the *last* estimate before the float actually surfaced.
    final = (timing_df.sort_values("lead_h")
                       .groupby(["float_id", "model", "real_t"], as_index=False)
                       .first())
    final = final.sort_values("real_t")
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(final) + 1.5))
    labels = [f"{r.float_id} / {r.model}\n{r.real_t:%m-%d %H:%M}" for r in final.itertuples()]
    colors = [MODEL_COLOR.get(m, "0.5") for m in final["model"]]
    bars = ax.barh(labels, final["timing_error_h"], color=colors, height=0.6)
    for bar, r in zip(bars, final.itertuples()):
        ax.text(bar.get_width() + np.sign(bar.get_width() or 1) * 0.3, bar.get_y() + bar.get_height() / 2,
                f"{r.timing_error_h:+.1f}h (lead {r.lead_h:.1f}h)", va="center", fontsize=8)
    ax.axvline(0, color="0.3", linewidth=1)
    ax.set_xlabel("Final predicted surfacing time minus actual (hours)")
    ax.set_title("Last forecast before each real surfacing: how far off on timing")
    fig.tight_layout()
    plt.show()


## 4. Depth: simulated dive profile vs. real pressure trace

Two independent depth signals for the same dive:

- **Simulated** -- reconstructed from the leg's `cycle_action`
  (descent/parking/ascent speeds and target depth) using the exact same
  phase-recovery formula `simulate.simulate_cycle` uses internally (depth
  doesn't depend on ocean currents, so this needs no model data). The
  `cycle_action` used is whichever `cycle_action_history` snapshot was
  most recent as of this surfacing -- an approximation, since the action
  estimate can itself change mid-leg as `run._refresh_cycle_actions` reruns;
  this uses the estimate as of scoring time, not a mid-leg-accurate splice.
- **Real** -- the float's own recorded `PRES` (dbar) readings during that
  dive, read directly from the cached `{float_id}_Rtraj.nc` in
  `data/argo_cache/` (matched to this surfacing via `JULD_LAST_LOCATION`,
  within a tolerance -- see `DEPTH_MATCH_TOLERANCE_H` below). That cache is
  gitignored and only as fresh as the last time something on this machine
  ran the pipeline or refreshed it -- legs with no sufficiently-fresh match
  are skipped with a stated reason rather than silently plotted wrong.

y-axis is inverted (depth increases downward) to match oceanographic
convention.


In [ ]:
DEPTH_MATCH_TOLERANCE_H = 12.0  # widen if your argo_cache is a bit stale and you still want a (labeled) comparison

def _simulated_depth_series(ca: ControlAction, anchor_time, until_time, dt_s: float = 1800.0):
    """(times, depths) reconstructed via simulate_cycle's own phase formula -- see
    ControlAction's docstring: cycle_hours is descent+parking only, so the full
    repeat period is reconstructed from it plus ascent/transmission time."""
    target_depth = ca.target_depth or 0.0
    descent_s = target_depth / ca.descent_speed_ms if target_depth > 0 else 0.0
    ascent_s  = target_depth / ca.ascent_speed_ms  if target_depth > 0 else 0.0
    transmission_s = ca.transmission_duration_minutes * 60.0
    descent_plus_parking_s = ca.cycle_hours * 3600.0
    parking_s = max(descent_plus_parking_s - descent_s, 0.0)
    total_cycle_s = descent_plus_parking_s + ascent_s + transmission_s

    ts, depths = [], []
    t, elapsed = anchor_time, 0.0
    total_s = (until_time - anchor_time).total_seconds()
    while elapsed <= total_s:
        cycle_elapsed = elapsed % total_cycle_s
        if cycle_elapsed < descent_s:
            depth = ca.descent_speed_ms * cycle_elapsed
        elif cycle_elapsed < descent_s + parking_s:
            depth = target_depth
        elif cycle_elapsed < descent_s + parking_s + ascent_s:
            depth = max(target_depth - ca.ascent_speed_ms * (cycle_elapsed - descent_s - parking_s), 0.0)
        else:
            depth = 0.0
        ts.append(t); depths.append(depth)
        t += timedelta(seconds=dt_s)
        elapsed += dt_s
    return ts, depths


# Measurement codes covering descent/parking/ascent PRES readings -- see
# cycle_extractor.py's DESC_CODE/PARKING_CODE/ASC_CODE plus their
# sub-phase siblings (drift-at-park variants, deep-descent/ascent).
_REAL_DEPTH_CODES = {190, 198, 203, 250, 289, 290, 297, 298, 590}


def _real_depth_series(float_id: str, cache_dir: Path, near_time, tolerance_h: float):
    """Real (t, PRES) trace for whichever cached cycle's JULD_LAST_LOCATION is
    closest to `near_time`, or (None, reason) if no cache file exists or the
    closest cycle is further than `tolerance_h` away (stale cache)."""
    rtraj_path = Path(cache_dir) / f"{float_id}_Rtraj.nc"
    if not rtraj_path.exists():
        return None, "no cached Rtraj.nc for this float in data/argo_cache/"

    ds = xr.open_dataset(rtraj_path)
    try:
        last_loc = pd.to_datetime(ds["JULD_LAST_LOCATION"].values)
        cyc_idx = ds["CYCLE_NUMBER_INDEX"].values.astype(int)
        deltas_h = (last_loc - pd.Timestamp(near_time)) / pd.Timedelta(hours=1)
        i = int(np.argmin(np.abs(deltas_h)))
        if abs(deltas_h[i]) > tolerance_h:
            return None, (
                f"nearest cached real cycle is {abs(deltas_h[i]):.0f}h from this surfacing "
                f"(cache is stale -- rerun the pipeline to refresh {rtraj_path.name})"
            )
        cyc = cyc_idx[i]
        mask = (ds["CYCLE_NUMBER"].values == cyc) & np.isin(ds["MEASUREMENT_CODE"].values, list(_REAL_DEPTH_CODES))
        juld = ds["JULD"].values[mask]
        pres = ds["PRES"].values[mask]
        park_pressure = float(ds["REPRESENTATIVE_PARK_PRESSURE"].values[i])
    finally:
        ds.close()

    valid = ~pd.isna(juld) & ~np.isnan(pres)
    juld, pres = juld[valid], pres[valid]
    if len(juld) == 0:
        return None, f"cycle {cyc} matched but has no valid PRES readings"
    order = np.argsort(juld)
    return dict(t=pd.to_datetime(juld[order]), depth=pres[order], cycle=int(cyc),
                park_pressure=park_pressure), None


In [ ]:
depth_legs = []  # accumulate (float_id, model, real_t, sim_t, sim_d, real_or_none, skip_reason)

for r in scoring.itertuples():
    leg_pts = leg_history_db[
        (leg_history_db["float_id"] == r.float_id) & (leg_history_db["model"] == r.model)
        & (pd.to_datetime(leg_history_db["leg_end_time"]) == r.t)
    ]
    if leg_pts.empty:
        depth_legs.append((r.float_id, r.model, r.t, None, None, None,
                            "no leg_history rows for this leg (older leg, full path wasn't retained then)"))
        continue
    anchor_time = pd.to_datetime(leg_pts["t"]).min().to_pydatetime()

    cah = cycle_action_history_db[cycle_action_history_db["float_id"] == r.float_id].copy()
    cah["logged_at"] = pd.to_datetime(cah["logged_at"])
    cah = cah[cah["logged_at"] <= r.t].sort_values("logged_at")
    if cah.empty:
        depth_legs.append((r.float_id, r.model, r.t, None, None, None,
                            "no cycle_action_history snapshot at/before this surfacing"))
        continue
    ca_row = cah.iloc[-1]
    ca = ControlAction(
        park_mode=ca_row["park_mode"], cycle_hours=ca_row["cycle_hours"],
        transmission_duration_minutes=ca_row["transmission_duration_minutes"],
        target_depth=ca_row["target_depth"], descent_speed_ms=ca_row["descent_speed_ms"],
        ascent_speed_ms=ca_row["ascent_speed_ms"],
    )
    sim_t, sim_d = _simulated_depth_series(ca, anchor_time, r.t.to_pydatetime())
    real, reason = _real_depth_series(r.float_id, ARGO_CACHE_DIR, r.t, DEPTH_MATCH_TOLERANCE_H)
    depth_legs.append((r.float_id, r.model, r.t, sim_t, sim_d, real, reason))

n_with_real = sum(1 for *_, real, _ in depth_legs if real is not None)
print(f"{len(depth_legs)} scored legs total, {n_with_real} have a fresh-enough real depth trace to overlay")
for float_id, model, t, sim_t, sim_d, real, reason in depth_legs:
    status = f"real cycle {real['cycle']} matched" if real is not None else f"skipped -- {reason}"
    print(f"  {float_id} / {model} / {t:%Y-%m-%d %H:%M}: {status}")


In [ ]:
plottable = [d for d in depth_legs if d[3] is not None]  # has a simulated series at least

if not plottable:
    print("Nothing to plot -- no leg has both leg_history and a cycle_action_history snapshot.")
else:
    ncols = 3
    nrows = -(-len(plottable) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.3 * nrows), squeeze=False)

    for idx, (float_id, model, t, sim_t, sim_d, real, reason) in enumerate(plottable):
        ax = axes[idx // ncols][idx % ncols]
        ax.plot(sim_t, sim_d, "--", color=MODEL_COLOR.get(model, "0.5"),
                label=f"simulated ({web_export.MODEL_DISPLAY.get(model, model)})", linewidth=1.6)
        if real is not None:
            ax.plot(real["t"], real["depth"], "-o", color="0.15", label="real (Rtraj PRES)",
                     markersize=3, linewidth=1.2)
            ax.axhline(real["park_pressure"], color="0.15", linestyle=":", linewidth=0.8, alpha=0.6)
        else:
            ax.text(0.5, 0.55, "no fresh real trace\n(" + reason[:40] + "...)" if len(reason) > 40 else f"no fresh real trace\n({reason})",
                    transform=ax.transAxes, ha="center", va="center", fontsize=6.5, color="0.4", wrap=True)

        ax.invert_yaxis()
        ax.set_title(f"{float_id} / {model}\n{t:%Y-%m-%d %H:%M}", fontsize=9)
        ax.set_ylabel("depth (dbar)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6.5, loc="lower right")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))

    for idx in range(len(plottable), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    fig.suptitle("Simulated dive profile vs. real pressure trace, per scored leg", fontsize=10)
    fig.tight_layout()
    plt.show()


## Notes / caveats

- **Small sample.** As of whenever you run this, section 1's table is
  exactly `len(scoring)` rows -- this leaderboard is early, so expect a
  handful of events, not hundreds. Don't over-read a single 100%-of-drift
  event as "the model is bad."
- **`error_pct` needs `drift_m >= web_export.MIN_DRIFT_M_FOR_PCT`** (100 m)
  -- below that the float barely moved and the ratio is dominated by GPS
  noise, so it's left blank rather than shown as a misleadingly huge or
  tiny number.
- **Timing section only uses forecasts issued strictly inside a completed
  leg** (`prev_real_t <= issue_time < real_t`) -- a forecast issued before
  the previous surfacing belongs to an earlier leg and is excluded, even
  if its `expected_surfacing_time` happens to fall in this window.
- **Depth section's `cycle_action` is an approximation** -- it's whichever
  `cycle_action_history` snapshot was current as of the surfacing that
  closed the leg, not a mid-leg-accurate reconstruction of what
  `run.py` actually simulated with at each point along the way (the action
  estimate can itself shift between pipeline runs within a single leg).
- **Depth section's real trace depends entirely on `data/argo_cache/`
  freshness**, which this notebook does not fetch itself (network access
  is a pipeline concern, not a diagnostics-notebook one) -- rerun
  `python main.py` first if you want maximum coverage here.
